In [ ]:
import os
import pdfplumber
import pandas as pd

# ---------- INPUT / OUTPUT PATH ----------
pdf_path = r"C:\Users\Sohel\Desktop\1450_DNCC_Bill_Pay_0406202613202800659.pdf"
excel_path = r"C:\Users\Sohel\Desktop\1450.xlsx"  

# ---------- PROCESS ----------
output_rows = []

with pdfplumber.open(pdf_path) as pdf:
    total_pages = len(pdf.pages)
    for page_num, page in enumerate(pdf.pages, start=1):
        try:
            table = page.extract_table()
            if not table:
                continue
            for row in table:
                if (
                    not row
                    or (row[0] and "Division" in row[0])
                    or (row[0] and "© DBBL" in row[0])
                    or (row[0] and "Page" in row[0])
                ):
                    continue
                row = [cell.replace("\n", "").strip() if cell else "" for cell in row]
                if len(row) == 17:
                    output_rows.append(row)
        except Exception as e:
            print(f"Error processing page {page_num}: {e}")
            continue

if not output_rows:
    print("No valid data found in PDF.")
else:
    columns = [
        "Division", "District", "Territory", "Territory Code", "Txn Date",
        "API Name", "Txn ID", "Txn Acc", "Bill No", "Trace No",
        "Subagent Casa", "Subagent Name", "Agent Casa", "Agent Name",
        "Amount", "Fee Amt"
    ]

    df = pd.DataFrame([row[:-1] for row in output_rows], columns=columns)
    df.to_excel(excel_path, index=False)
    print(f"File saved at: {excel_path}")